In [0]:
from pyspark.sql import functions as F

BRONZE_PATH = "/Volumes/workspace/drift_monitor/bronze/transactions/"
SILVER_TABLE = "workspace.drift_monitor.silver_transactions"
OUTGOING = ["CASH_OUT", "PAYMENT", "TRANSFER", "DEBIT"]

In [0]:
print(repr(BRONZE_PATH))
files = dbutils.fs.ls(BRONZE_PATH)
print(len(files))
display(files[:5])

In [0]:
silver = (
    spark.read.parquet(BRONZE_PATH)
    .withColumn("is_outgoing", F.col("transaction_type").isin(OUTGOING))
    .withColumn(
        "balance_mismatch",
        F.col("is_outgoing")
        & (F.abs(F.col("origin_balance_before") - F.col("amount") - F.col("origin_balance_after")) > 0.01),
    )
    .withColumn("day", ((F.col("step") - 1) / 24).cast("int") + 1)
    .withColumn("hour_of_day", (F.col("step") - 1) % 24)
)

silver.write.mode("overwrite").saveAsTable(SILVER_TABLE)

verify


In [0]:
%sql
SELECT
    COUNT(*)                           AS total_rows,
    SUM(CAST(balance_mismatch AS INT)) AS mismatches,
    MIN(day)                           AS first_day,
    MAX(day)                           AS last_day
FROM workspace.drift_monitor.silver_transactions

In [0]:
%sql
DESCRIBE DETAIL workspace.drift_monitor.silver_transactions

In [0]:
detail = spark.sql(f"DESCRIBE DETAIL {SILVER_TABLE}")
display(
    detail.select(
        "format",
        "numFiles",
        F.round(F.col("sizeInBytes") / 1024 / 1024, 1).alias("size_mb"),
    )
)